In [1]:
import sys
sys.path.append("../")

In [2]:
from typing import List, Optional
import requests
import pandas as pd
import mygene
from utility import resolve_ensembl_ids_with_fallback, get_chembl_ids_fast, get_disease_ids_fast, validate_id_dataframe
import pickle

In [3]:
dct_result = dict()

In [4]:
with open("Opentargets_same_question_response.pkl", "rb") as f:
    Opentargets_same_question_response = pickle.load(f)


In [5]:
# Opentargets_same_question_response

In [6]:
for model in ['llama-3.3-70b-versatile', 'gpt-5-nano', 'grok-4-1-fast-non-reasoning-latest', "gemini-2.5-flash-lite", "OpenTargets", "BioChatter"]:

    
    # if model=="llama-3.3-70b-versatile":
    #     dct_result[model] = Llama_same_question_response[model]

    # if model=="gpt-5-nano":
    #     dct_result[model] = OpenAI_same_question_response[model]

    # if model=='grok-4-1-fast-non-reasoning-latest':
    #     dct_result[model] = Grok_same_question_response[model]

    # if model=="gemini-2.5-flash-lite":
    #     dct_result[model] = Gemini_same_question_response[model]

    if model=="OpenTargets":
        dct_result[model] = Opentargets_same_question_response[model]

    # if model=="BioChatter":
    #     dct_result[model] = Biochatter_same_question_response[model]


In [7]:
# dct_result

In [8]:
allowed = {"gene_name", "drug_name", "disease_name", "pathway_name", "gene_id", "drug_id", 'gene_symbol', 
"disease_id", "pathway_id", "action_types", "evidence", "source_database", "association_score", 
"mechanism_of_action", "phase", "status", "top_level_term","target_name", "drug_type", "indication_id",
 "indication_name", "target_id", "target_ensembl_id", "drug_chembl_id", "disease_efo_id", "references"}

missing_df = []
not_dataframe = []
empty_df = []
bad_columns = []

total_payloads = 0

for model, queries in dct_result.items():
    if not isinstance(queries, dict):
        continue

    for question, runs in queries.items():
        if not isinstance(runs, dict):
            continue

        for run_id, payload in runs.items():
            total_payloads += 1

            if not isinstance(payload, dict) or "dataframe" not in payload:
                missing_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "payload_type": type(payload).__name__
                })
                continue

            df = payload.get("dataframe")

            if not isinstance(df, pd.DataFrame):
                not_dataframe.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "df_type": type(df).__name__
                })
                continue

            if df.empty:
                empty_df.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns)
                })

            cols = {str(c) for c in df.columns}
            extra = sorted(cols - allowed)

            if extra:
                bad_columns.append({
                    "model": model,
                    "question": question,
                    "run_id": run_id,
                    "columns": list(df.columns),
                    "extra_columns": extra
                })

print(f"Total payloads checked: {total_payloads}")
print(f"Missing dataframe key: {len(missing_df)}")
print(f"Dataframe is not pd.DataFrame: {len(not_dataframe)}")
print(f"Empty dataframes: {len(empty_df)}")
print(f"Dataframes with disallowed columns: {len(bad_columns)}")

bad_columns_df = pd.DataFrame(bad_columns)
missing_df_df = pd.DataFrame(missing_df)
not_dataframe_df = pd.DataFrame(not_dataframe)
empty_df_df = pd.DataFrame(empty_df)

display(bad_columns_df)
display(missing_df_df)
display(not_dataframe_df)
display(empty_df_df)


Total payloads checked: 349
Missing dataframe key: 0
Dataframe is not pd.DataFrame: 0
Empty dataframes: 0
Dataframes with disallowed columns: 0


""


""


""


""


In [9]:
dct_jaccard = {}

# GLOBAL entity store (single dictionary)
question_entities = {
    "gene_name": set(),
    "drug_name": set(),
    "disease_name": set(),
}

allowed_cols = {"gene_name", "drug_name", "disease_name"}

for model, queries in dct_result.items():
    dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question, runs in queries.items():

        for run_id, payload in runs.items():
            df = payload.get("dataframe")

            # ---- validation ----
            if not isinstance(df, pd.DataFrame) or df.empty:
                continue

            # ---- skip pathway outputs ----
            if "pathway_name" in df.columns:
                continue

            # ---- column sanity ----
            if not set(df.columns).issubset(allowed_cols):
                continue

            # ---- collect entities globally ----
            for col in allowed_cols:
                if col in df.columns:
                    values = (
                        df[col]
                        .dropna()
                        .astype(str)
                        .str.strip()
                        .tolist()
                    )
                    question_entities[col].update(values)

# ---- finalize (sets → sorted lists) ----
for k in question_entities:
    question_entities[k] = sorted(question_entities[k])


Working for model: OpenTargets


##### Extract all gene entry

In [10]:
df_gene = resolve_ensembl_ids_with_fallback(list(question_entities["gene_name"]),use_opentargets_fallback=True)
df_gene.drop_duplicates()
df_gene

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


KeyError: "None of ['query'] are in the columns"

In [ ]:
df_gene[df_gene.isna().any(axis=1)]

##### Associate id with all drug entry

In [ ]:
df_drug = await get_chembl_ids_fast(list(question_entities["drug_name"]))
df_drug.drop_duplicates()
df_drug

In [ ]:
df_drug[df_drug.isna().any(axis=1)]

##### Associate id with all disease entry

In [ ]:
df_disease = await get_disease_ids_fast(list(question_entities["disease_name"]))
df_disease.drop_duplicates()
# df_disease.drop_duplicates()


In [ ]:
df_disease[df_disease.isna().any(axis=1)]

In [ ]:
df_disease["source"].value_counts()

In [ ]:
# def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
#     """
#     Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

#     For each (model, question):
#     1) Keep only valid dataframe runs with at least one entity column.
#     2) Map gene/drug/disease names to canonical IDs using normalized join keys.
#     3) Build one set of ID tuples per run.
#     4) Compute Jaccard = |intersection(all runs)| / |union(all runs)|.

#     Side effect:
#     - Stores the mapped dataframe in dct_result[model][question][run]['dataframe_id'].

#     Returns:
#     - dct_jaccard: dict[model][question] -> float
#     """

#     dct_jaccard = {}
#     entity_cols = {"gene_name", "drug_name", "disease_name"}

#     # Build compact, deterministic mapping tables once (avoids repeated many-to-many merges).
#     gene_map = (
#         df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
#         [["_gene_norm", "gene_id"]]
#         .dropna(subset=["gene_id"])
#         .drop_duplicates(subset=["_gene_norm"], keep="first")
#     )
#     disease_map = (
#         df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
#         [["_disease_norm", "disease_id"]]
#         .dropna(subset=["disease_id"])
#         .drop_duplicates(subset=["_disease_norm"], keep="first")
#     )
#     drug_map = (
#         df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
#         [["_drug_norm", "drug_id"]]
#         .dropna(subset=["drug_id"])
#         .drop_duplicates(subset=["_drug_norm"], keep="first")
#     )

#     for model, model_payload in dct_result.items():
#         dct_jaccard[model] = {}
#         print(f"\nWorking for model: {model}")

#         for question_key, runs_dict in model_payload.items():
#             tmp_result = {}

#             for run_number, run_payload in runs_dict.items():
#                 df = run_payload.get("dataframe")

#                 # Basic validity checks.
#                 if not isinstance(df, pd.DataFrame) or df.empty:
#                     continue

#                 # Skip pathway outputs; this cell evaluates only gene/drug/disease outputs.
#                 if "pathway_name" in df.columns:
#                     continue

#                 present_entities = [c for c in ["gene_name", "disease_name", "drug_name"] if c in df.columns]
#                 if not present_entities:
#                     continue

#                 work_df = df.copy()
#                 final_columns = []

#                 # Gene name -> gene_id (left join keeps original run rows).
#                 if "gene_name" in work_df.columns:
#                     work_df["_gene_norm"] = work_df["gene_name"].astype(str).str.strip().str.upper()
#                     work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
#                     final_columns.append("gene_id")

#                 # Disease name -> disease_id.
#                 if "disease_name" in work_df.columns:
#                     work_df["_disease_norm"] = work_df["disease_name"].astype(str).str.strip().str.upper()
#                     work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
#                     final_columns.append("disease_id")

#                 # Drug name -> drug_id.
#                 if "drug_name" in work_df.columns:
#                     work_df["_drug_norm"] = work_df["drug_name"].astype(str).str.strip().str.upper()
#                     work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
#                     final_columns.append("drug_id")

#                 # Keep only canonical ID columns for set-based run comparison.
#                 if not final_columns:
#                     continue

#                 id_df = (
#                     work_df[final_columns]
#                     .drop_duplicates()
#                     .dropna(how="any")
#                     .reset_index(drop=True)
#                 )

#                 # Save mapped IDs for debugging/inspection.
#                 dct_result[model][question_key][run_number]["dataframe_id"] = id_df

#                 # Treat empty mapped outputs as invalid runs for Jaccard.
#                 if id_df.empty:
#                     continue

#                 run_set = {
#                     tuple(str(v) for v in row)
#                     for row in id_df.itertuples(index=False, name=None)
#                 }

#                 if not run_set:
#                     continue

#                 tmp_result[run_number] = run_set

#             valid_runs = len(tmp_result)
#             expected_runs = len(runs_dict)

#             if valid_runs < 2:
#                 print(
#                     f"Not enough valid runs for '{question_key}' "
#                     f"({valid_runs}/{expected_runs}). Skipping Jaccard."
#                 )
#                 continue

#             if valid_runs < expected_runs:
#                 print(
#                     f"Warning: partial runs valid for '{question_key}': "
#                     f"{valid_runs}/{expected_runs}"
#                 )

#             sets = list(tmp_result.values())
#             intersection = set.intersection(*sets)
#             union = set.union(*sets)

#             if not union:
#                 print(f"Empty union for '{question_key}', skipping Jaccard.")
#                 continue

#             jaccard_index = len(intersection) / len(union)
#             dct_jaccard[model][question_key] = jaccard_index

#             print(
#                 f"Jaccard index for '{question_key}': "
#                 f"{jaccard_index:.4f}"
#             )

#     return dct_jaccard


# dct_jaccard = compute_jaccard_from_run_dataframes(
#     dct_result=dct_result,
#     df_gene=df_gene,
#     df_disease=df_disease,
#     df_drug=df_drug,
# )

In [ ]:
def compute_jaccard_from_run_dataframes(dct_result, df_gene, df_disease, df_drug):
    """
    Compute per-question cross-run Jaccard consistency after mapping entities to IDs.

    For each (model, question):
    - intersection: entries present in ALL valid runs
    - union: entries present in ANY valid run
    - jaccard = |intersection| / |union|

    All comparisons are CASE-INDEPENDENT and ID-based.

    Side effects:
    - Stores per-run mapped dataframe in:
        dct_result[model][question][run]['dataframe_id']

    Returns:
    - dct_jaccard[model][question] = {
          'jaccard': float,
          'n_valid_runs': int,
          'intersection': set[tuple],
          'union': set[tuple],
      }
    """

    dct_jaccard = {}

    # ------------------------------------------------------------------
    # Build deterministic one-to-one mapping tables
    # ------------------------------------------------------------------
    gene_map = (
        df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
        [["_gene_norm", "gene_id"]]
        .dropna(subset=["gene_id"])
        .drop_duplicates("_gene_norm", keep="first")
    )

    disease_map = (
        df_disease.assign(_disease_norm=df_disease["disease_name"].astype(str).str.strip().str.upper())
        [["_disease_norm", "disease_id"]]
        .dropna(subset=["disease_id"])
        .drop_duplicates("_disease_norm", keep="first")
    )

    drug_map = (
        df_drug.assign(_drug_norm=df_drug["drug_name"].astype(str).str.strip().str.upper())
        [["_drug_norm", "drug_id"]]
        .dropna(subset=["drug_id"])
        .drop_duplicates("_drug_norm", keep="first")
    )

    # ------------------------------------------------------------------
    # Main loop
    # ------------------------------------------------------------------
    for model, model_payload in dct_result.items():
        dct_jaccard[model] = {}
        print(f"\nWorking for model: {model}")

        for question_key, runs_dict in model_payload.items():
            run_sets = {}

            for run_number, run_payload in runs_dict.items():
                df = run_payload.get("dataframe")

                # ---- basic validity ----
                if not isinstance(df, pd.DataFrame) or df.empty:
                    continue

                if "pathway_name" in df.columns:
                    continue

                work_df = df.copy()
                final_cols = []

                # ---- gene ----
                if "gene_name" in work_df.columns:
                    work_df["_gene_norm"] = (
                        work_df["gene_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(gene_map, on="_gene_norm", how="left")
                    final_cols.append("gene_id")

                # ---- disease ----
                if "disease_name" in work_df.columns:
                    work_df["_disease_norm"] = (
                        work_df["disease_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(disease_map, on="_disease_norm", how="left")
                    final_cols.append("disease_id")

                # ---- drug ----
                if "drug_name" in work_df.columns:
                    work_df["_drug_norm"] = (
                        work_df["drug_name"].astype(str).str.strip().str.upper()
                    )
                    work_df = work_df.merge(drug_map, on="_drug_norm", how="left")
                    final_cols.append("drug_id")

                if not final_cols:
                    continue

                # ---- canonical ID dataframe ----
                id_df = (
                    work_df[final_cols]
                    .dropna(how="any")
                    .drop_duplicates()
                    .reset_index(drop=True)
                )

                dct_result[model][question_key][run_number]["dataframe_id"] = id_df

                if id_df.empty:
                    continue

                # ---- CASE-INDEPENDENT tuple set ----
                run_set = {
                    tuple(str(v).strip().upper() for v in row)
                    for row in id_df.itertuples(index=False, name=None)
                }

                if run_set:
                    run_sets[run_number] = run_set

            # ------------------------------------------------------------------
            # Compute intersection / union
            # ------------------------------------------------------------------
            n_valid_runs = len(run_sets)

            if n_valid_runs < 2:
                print(
                    f"Not enough valid runs for '{question_key}' "
                    f"({n_valid_runs}/{len(runs_dict)}). Skipping."
                )
                continue

            sets = list(run_sets.values())
            intersection = set.intersection(*sets)
            union = set.union(*sets)

            if not union:
                print(f"Empty union for '{question_key}', skipping.")
                continue

            jaccard = len(intersection) / len(union)

            dct_jaccard[model][question_key] = {
                "jaccard": jaccard,
                "n_valid_runs": n_valid_runs,
                "intersection": intersection,
                "union": union,
            }

            print(
                f"'{question_key}': "
                f"J={jaccard:.4f}, "
                f"|∩|={len(intersection)}, "
                f"|∪|={len(union)}, "
                f"runs={n_valid_runs}"
            )

    return dct_jaccard


dct_jaccard = compute_jaccard_from_run_dataframes(
    dct_result=dct_result,
    df_gene=df_gene,
    df_disease=df_disease,
    df_drug=df_drug,
)

In [ ]:
for model, model_payload in dct_result.items():
    # dct_jaccard[model] = {}
    print(f"\nWorking for model: {model}")

    for question_key, runs_dict in model_payload.items():
        run_sets = {}

        for run_number, run_payload in runs_dict.items():
            # df = run_payload.get("dataframe_id")
            df = run_payload.get("dataframe_id")
            is_valid = validate_id_dataframe(
                df,
                model=model,
                question_key=question_key,
                run_number=run_number,
            )
            if not is_valid:
                print(f"Invalid dataframe for model={model}, question='{question_key}', run={run_number}. Skipping.")
                continue

            run_sets[run_number] = set(map(tuple, df.values))

In [ ]:
df

In [ ]:
# df

In [ ]:
# with open("Grok_same_question_response_with_id.pkl", "wb") as f:
#     pickle.dump(dct_result, f)

In [ ]:
q = 'Which pathways are associated with the JAK2 gene?'
model = 'OpenTarget'

In [ ]:
# dct_result['grok-4-1-fast-non-reasoning-latest']

In [ ]:
dct_result[model][q][1]['dataframe']

In [ ]:
dct_result[model][q][2]['dataframe']

In [ ]:
dct_result[model][q][3]['dataframe']

In [ ]:
dct_result[model][q][4]['dataframe']

In [ ]:
dct_result[model][q][5]['dataframe']

In [ ]:
dct_result[model][q][4]['dataframe_id']

In [ ]:
# gene_map = (
#     df_gene.assign(_gene_norm=df_gene["gene_name"].astype(str).str.strip().str.upper())
#     [["_gene_norm", "gene_id"]]
#     .dropna(subset=["gene_id"])
#     .drop_duplicates("_gene_norm", keep="first")
# )

In [ ]:
# gene_map

In [ ]:
with open("Opentargets_same_question_response_with_id.pkl", "wb") as f:
    pickle.dump(dct_result, f)